In [1]:
import sys
import torch
import transformers

print("Python:")
print(sys.executable)

print("\nPyTorch:")
print(torch.__version__)

print("\nCUDA:")
print(torch.cuda.is_available())
print(torch.version.cuda)

print("\nGPU:")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

print("\nTransformers:")
print(transformers.__version__)

c:\Users\Dmytro\Desktop\python\UkrainianSmallTransformer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python:
c:\Users\Dmytro\Desktop\python\UkrainianSmallTransformer\.venv\Scripts\python.exe

PyTorch:
2.14.0+cu130

CUDA:
True
13.0

GPU:
NVIDIA GeForce RTX 4060 Laptop GPU

Transformers:
5.17.0


In [2]:
from create_model import create_model
import numpy as np
import time
from pathlib import Path

In [3]:
ALL_TOKENS = 500_000_000
BATCH_SIZE = 32
SEQ_LEN = 256

TRAIN_TOKENS = 100_000_000

MAX_LR = 1e-3
MIN_LR = 1e-4
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.1

EVAL_EVERY = 250
EVAL_BATCHES = 50
MODEL_SIZE = "3M"
MODEL_NAME = f"{MODEL_SIZE}_model_on_{TRAIN_TOKENS // 1000000.0}M_tokens"
MODEL_DIR = Path("models")

In [34]:
train_tokens = np.memmap(
    "data/train_500M.bin",
    dtype=np.uint16,
    mode="r",
    shape=(TRAIN_TOKENS,),
)

val_tokens = np.memmap(
    "data/validation/validation_5M.bin",
    dtype=np.uint16,
    mode="r",
    shape=(5_000_000,),
)
test_tokens = np.memmap(
    "data/test/test_5M.bin",
    dtype=np.uint16,
    mode="r",
    shape=(5_000_000,),
)

In [35]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = create_model(MODEL_SIZE)
model = model.to(device)

print(device)

cuda


In [36]:
def get_batch(
    tokens,
    batch_size,
    seq_len,
    device,
):
    max_start = len(tokens) - seq_len - 1

    starts = np.random.randint(
        0,
        max_start,
        size=batch_size,
    )

    x = np.stack([
        tokens[i:i + seq_len]
        for i in starts
    ])

    y = np.stack([
        tokens[i + 1:i + seq_len + 1]
        for i in starts
    ])

    x = torch.from_numpy(
        x.astype(np.int64)
    )

    y = torch.from_numpy(
        y.astype(np.int64)
    )

    x = x.to(device)
    y = y.to(device)

    return x, y

In [37]:
import math

tokens_per_step = BATCH_SIZE * SEQ_LEN

max_steps = math.ceil(
    TRAIN_TOKENS / tokens_per_step
)

print("Tokens per step:", tokens_per_step)
print("Max steps:", max_steps)

Tokens per step: 8192
Max steps: 1221


In [38]:
@torch.no_grad()
def evaluate(
    model,
    tokens,
    batch_size,
    seq_len,
    device,
    num_batches=50,
):
    model.eval()

    losses = []

    for _ in range(num_batches):
        x, y = get_batch(
            tokens=tokens,
            batch_size=batch_size,
            seq_len=seq_len,
            device=device,
        )

        outputs = model(
            input_ids=x,
            labels=x,
        )

        losses.append(
            outputs.loss.item()
        )

    model.train()

    return float(np.mean(losses))

In [39]:
def get_lr(step, max_steps):
    if step < WARMUP_STEPS:
        return MAX_LR * step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, max_steps - WARMUP_STEPS)
    return MIN_LR + 0.5 * (MAX_LR - MIN_LR) * (1 + math.cos(math.pi * progress))

In [40]:
def train(model, train_tokens, val_tokens, device):
    decay = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.requires_grad and p.dim() < 2]

    optimizer = torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=MAX_LR,
        betas=(0.9, 0.95),
    )

    tokens_per_step = BATCH_SIZE * SEQ_LEN
    max_steps = math.ceil(TRAIN_TOKENS / tokens_per_step)
    best_val_loss = float("inf")

    model.train()
    start_time = time.time()

    for step in range(1, max_steps + 1):

        x, y = get_batch(
            tokens=train_tokens,
            batch_size=BATCH_SIZE,
            seq_len=SEQ_LEN,
            device=device,
        )

        lr = get_lr(step, max_steps)
        for g in optimizer.param_groups:
            g["lr"] = lr

        optimizer.zero_grad(set_to_none=True)

        outputs = model(input_ids=x, labels=x)
        loss = outputs.loss

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        if step % 50 == 0:
            tokens_seen = step * tokens_per_step
            elapsed = time.time() - start_time
            print(
                f"step={step:5d} "
                f"tokens={tokens_seen / 1e6:8.2f}M "
                f"loss={loss.item():.4f} "
                f"lr={lr:.2e} "
                f"gnorm={grad_norm:.2f} "
                f"time={elapsed:.1f}s"
            )

        if step % EVAL_EVERY == 0 or step == max_steps:

            val_loss = evaluate(
                model=model,
                tokens=val_tokens,
                batch_size=BATCH_SIZE,
                seq_len=SEQ_LEN,
                device=device,
                num_batches=EVAL_BATCHES,
            )

            perplexity = math.exp(min(val_loss, 20))

            print(
                f"VALIDATION | step={step} "
                f"loss={val_loss:.4f} ppl={perplexity:.2f}"
            )

            if val_loss < best_val_loss:
                best_val_loss = val_loss

                torch.save(
                    {
                        "model": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "step": step,
                        "val_loss": val_loss,
                    },
                    MODEL_DIR / f"{MODEL_NAME}.pt",
                )

                print("Saved best model.")

In [41]:
train(
    model=model,
    train_tokens=train_tokens,
    val_tokens=val_tokens,
    device=device,
)

step=   50 tokens=    0.41M loss=8.5740 lr=2.50e-04 gnorm=0.90 time=9.3s
step=  100 tokens=    0.82M loss=7.8619 lr=5.00e-04 gnorm=0.45 time=18.5s
step=  150 tokens=    1.23M loss=7.5325 lr=7.50e-04 gnorm=0.79 time=27.8s
step=  200 tokens=    1.64M loss=7.1332 lr=1.00e-03 gnorm=0.50 time=37.2s
step=  250 tokens=    2.05M loss=6.8770 lr=9.95e-04 gnorm=0.79 time=46.6s
VALIDATION | step=250 loss=6.9926 ppl=1088.59
Saved best model.
step=  300 tokens=    2.46M loss=6.7604 lr=9.79e-04 gnorm=0.51 time=59.5s
step=  350 tokens=    2.87M loss=6.5846 lr=9.53e-04 gnorm=0.56 time=69.0s
step=  400 tokens=    3.28M loss=6.4494 lr=9.17e-04 gnorm=0.53 time=78.6s
step=  450 tokens=    3.69M loss=6.3657 lr=8.73e-04 gnorm=0.64 time=88.2s
step=  500 tokens=    4.10M loss=6.2181 lr=8.22e-04 gnorm=0.58 time=97.8s
VALIDATION | step=500 loss=6.3542 ppl=574.92
Saved best model.
step=  550 tokens=    4.51M loss=6.2502 lr=7.63e-04 gnorm=0.55 time=110.6s
step=  600 tokens=    4.92M loss=6.1026 lr=7.00e-04 gnorm=0

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = create_model(MODEL_SIZE)

checkpoint = torch.load(
    MODEL_DIR / f"{MODEL_NAME}.pt",
    map_location=device,
)

model.load_state_dict(checkpoint["model"])
model = model.to(device)
model.eval()

print("Loaded checkpoint")
print("Step:", checkpoint["step"])
print("Validation loss:", checkpoint["val_loss"])

Loaded checkpoint
Step: 1221
Validation loss: 5.750766925811767


In [43]:
import math


@torch.no_grad()
def evaluate_test(
    model,
    tokens,
    batch_size,
    seq_len,
    device,
):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    starts = range(
        0,
        len(tokens) - seq_len - 1,
        seq_len,
    )

    batch_x = []
    batch_y = []

    for start in starts:

        x = tokens[
            start:start + seq_len
        ]

        y = tokens[
            start + 1:start + seq_len + 1
        ]

        batch_x.append(x)
        batch_y.append(y)

        if len(batch_x) == batch_size:

            x = torch.from_numpy(
                np.stack(batch_x).astype(np.int64)
            ).to(device)

            y = torch.from_numpy(
                np.stack(batch_y).astype(np.int64)
            ).to(device)

            outputs = model(
                input_ids=x,
                labels=x,
            )

            loss = outputs.loss

            n = y.numel()

            total_loss += loss.item() * n
            total_tokens += n

            batch_x.clear()
            batch_y.clear()

    # leftover batch
    if batch_x:
        x = torch.from_numpy(
            np.stack(batch_x).astype(np.int64)
        ).to(device)

        y = torch.from_numpy(
            np.stack(batch_y).astype(np.int64)
        ).to(device)

        outputs = model(
            input_ids=x,
            labels=x,
        )

        loss = outputs.loss

        n = y.numel()

        total_loss += loss.item() * n
        total_tokens += n

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity

In [44]:
test_loss, test_ppl = evaluate_test(
    model=model,
    tokens=test_tokens,
    batch_size=32,
    seq_len=256,
    device=device,
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test perplexity: {test_ppl:.2f}")

Test loss: 5.7399
Test perplexity: 311.02


In [5]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(
    "tokenizer/tokenizer.json"
)

print(tokenizer.get_vocab_size())

16000


In [6]:
prompt = "Хвороба"

encoded = tokenizer.encode(prompt)

input_ids = torch.tensor(
    [encoded.ids],
    dtype=torch.long,
    device=device,
)
with torch.no_grad():
    output_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=100,

        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,

        eos_token_id=2,
        pad_token_id=3,
    )
    generated_text = tokenizer.decode(
    output_ids[0].tolist()
)

print(generated_text)

 Хворобачення в Україні «Нарамі», «Окрім того, не було право у них, а й не були. І, що це можна було було, а, і не було. Ми не в цілому, не зможемо, що у своєму цьому, то не було не було.
"Але в цей час, коли були, а й в цій ситуації.
Також, не є, тому, що в результаті рішення, він не варто сказати, чи не лише на нього,
